### Random vs Random

In [8]:
import time
from collections import Counter
from domain.configs import MAX_STEPS_PER_EPISODE
from environment.grenight_environment import GrenightEnvironment

In [9]:
NUM_GAMES = 1000

In [10]:
def play_random_game(env_arg: GrenightEnvironment) -> tuple[str, int, dict]:

    env_arg.reset()
    done = False
    move_count = 0
    acting_player_is_white = True
    reward = 0.0
    info = None
    pieces = None

    while not done and move_count < MAX_STEPS_PER_EPISODE:
        acting_player_is_white = env_arg.is_white_on_turn
        action = env_arg.sample()
        pieces = env_arg.pieces
        _, reward, done, info = env_arg.step(action)
        move_count += 1

    if not done:
        return "truncated", move_count, info
    if reward == 0.0:
        return "draw", move_count, info

    winner_is_white = acting_player_is_white if reward > 0 else not acting_player_is_white

    return ("white_win", move_count, info) if winner_is_white else ("black_win", move_count, info)

In [11]:
env = GrenightEnvironment()
outcomes_counter = Counter()
total_step_counts_per_outcome_counter = Counter()
draw_reasons_counter = Counter()

start_time = time.perf_counter()
for _ in range(NUM_GAMES):
    outcome, steps, game_info = play_random_game(env)
    outcomes_counter[outcome] += 1
    if outcome != "truncated":
        total_step_counts_per_outcome_counter[outcome] += steps
    if game_info["draw_reason"] is not None:
        draw_reasons_counter[game_info["draw_reason"]] += 1
end_time = time.perf_counter()

avg_step_counts_per_outcome = dict()
for outcome, total_steps in total_step_counts_per_outcome_counter.items():
    avg_step_counts_per_outcome[outcome] = total_steps / outcomes_counter[outcome]

print(f"STATS OUT FROM: {NUM_GAMES} GAMES\n"
      f"Outcomes: {outcomes_counter}\n"
      f"Average moves per outcome: {avg_step_counts_per_outcome}\n"
      f"Draw reasons: {draw_reasons_counter}\n"
      f"Execution time: {end_time - start_time:.2f} seconds\n")

STATS OUT FROM: 1000 GAMES
Outcomes: Counter({'draw': 702, 'black_win': 166, 'white_win': 132})
Average moves per outcome: {'draw': 73.84615384615384, 'black_win': 22.566265060240966, 'white_win': 24.454545454545453}
Draw reasons: Counter({'insufficient_material': 390, 'stalemate': 198, 'max_steps_without_progress': 68, 'threefold_repetition': 46})
Execution time: 54.19 seconds

